In [ ]:
import pandas as pd
import json
import os

In [ ]:
df_territories_city=pd.read_csv("~/Escritorio/Hito1/data/raw/datos_sucios_hito1.csv")
df_territories_city

In [ ]:
city_list=df_territories_city["Municipio"].unique().tolist()
city_list

In [ ]:
df_territories_accent=df_territories_city[df_territories_city["Región"].str.contains("á|é|í|ó|ú",case=False)]
list_accent=df_territories_accent["Región"].unique().tolist()
list_accent

In [ ]:
def accent_normalization(text):
    return (text.lower().strip()
            .replace("á","a")
            .replace("é","e")
            .replace("í","i")
            .replace("ó","o")
            .replace("ú","u"))
accent_normalization_dic={accent_normalization(a): a for a in list_accent}

def is_input_valid(text):
    if not text:
        return False
    return text.replace(" ","").isalpha()

In [ ]:
if os.path.exists("progress_territories.json"):
    with open ("progress_territories.json", "r") as f:
        raw_data_saved=json.load(f)
else:
    raw_data_saved={}
data_saved={k.strip():v for k,v in raw_data_saved.items()}

In [ ]:
def city_territories(city_list,progress=None):

    if progress is None:
        city_territories_dic={}
    else:
        city_territories_dic=progress.copy()
    
    for c in city_list:

        if c in city_territories_dic:
            continue

        while True:
            territories_raw=input(f"Ingrese una región para {c}: \n O ingrese '-' para terminar.").strip()
            
            if territories_raw=="-":
                return city_territories_dic
            
            if is_input_valid(territories_raw):
                break
            print ("La región solo puede contener letras.")


        #territories=accent_normalization.get(territories,territories.capitalize()) #El problema es que no hay separaciòn de datos
        territories_key=accent_normalization(territories_raw)
        territories_processed=accent_normalization_dic.get(territories_key,territories_raw.title())


        print(f"Municipio: {c}, región: {territories_processed}")
        city_territories_dic[c]=territories_processed

    return city_territories_dic

rpoint=city_territories(city_list, progress=data_saved)
with open ("progress_territories.json","w") as f:
    json.dump(rpoint,f)

print(json.dumps(rpoint, indent=4, ensure_ascii=False))

In [ ]:
#Voy a cargar el archivo original y el json. Pero aquí debo hacer un strip en municipio para que
#coincida con los valores del json. De lo contrario, no habría coincidencia
#Original
df_territories_city["Municipio"]=df_territories_city["Municipio"].str.strip()
df_territories_city

In [ ]:
#definitive_region_city=pd.read_json("./progress_territories.json",typ="series") #Si omite typ="series", no cargará el json
#pues técnicamente no es un dataframe, sino un diccionario. Pandas interpreta las llames como columnas y los valores como filas, pero ténicamente eso es un error porque no hay índices.
#de hecho, lo correcto sería usar la librería json

with open("progress_territories.json","r") as f:
    mapping_dict=json.load(f)
print(mapping_dict)

In [ ]:
#Aquí cruzo el diccionario con la tabla original. Debo recordar que el diccionario tiene como llave al municipio
#y que la región es el valor. Lo mismo debo hacer.
#es decir, en el archivo original, buscar la llave y reemplazar los valores. En este caso
#como no tengo condiciones lógicas, sino una referencia 1 a 1, lo mejor es usar map en vez de np.where
#pues este último sirve mucho para cuando tengo condiciones lógicas

df_territories_city["Región"]=df_territories_city["Municipio"].map(mapping_dict)
df_territories_city

In [ ]:
df_territories_city.to_csv("~/Escritorio/Hito1/datos_limpios.csv",index=False,encoding="utf-8")

Próxima tarea: que los datos se reemplacen efectivamente en el archivo. Crear una copia del archivo original para contrastar.